## 0 · Install dependencies

In [ ]:
!pip install pdfminer.six pymupdf opencv-python-headless numpy mistralai pydantic

## Config

In [13]:
import os

PDF_PATH = "OP-RAG.pdf"
# "pdfminer"  or  "mistral"
METHOD = "pdfminer"

MISTRAL_API_KEY = "2MdvgusXfBLLRE5C3BZpMDXcGI6UwcUA"
MISTRAL_PAGES   = list(range(8))

DPI   = 150
SCALE = DPI / 72.0

COLORS = {
    "text":   (255,  80,  80),   # red
    "image":  ( 50, 180,  50),   # green
    "figure": ( 50, 180,  50),   # green  (pdfminer only)
    "table":  ( 50, 130, 255),   # blue
    "graph":  (220, 150,   0),   # amber
    "line":   (180, 180, 180),   # grey   (pdfminer only)
}

OUTPUT_ROOT = "output"

print(f"Active method : {METHOD}")
print(f"Output folder : {os.path.join(OUTPUT_ROOT, METHOD)}")

Active method : pdfminer
Output folder : output\pdfminer


## Helper Functions

In [14]:
import cv2
import numpy as np


def draw_bbox(
    image: np.ndarray,
    location: tuple[int, int, int, int],
    label: str = "",
    color: tuple[int, int, int] = (255, 80, 80),
    thickness: int = 1,
) -> np.ndarray:

    x0, y0, x1, y1 = location
    cv2.rectangle(image, (x0, y0), (x1, y1), color, thickness)

    if label:
        font      = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.35
        (tw, th), _ = cv2.getTextSize(label, font, font_scale, 1)
        bg_y0 = max(y0 - th - 4, 0)
        bg_y1 = max(y0, th + 4)
        cv2.rectangle(image, (x0, bg_y0), (x0 + tw + 4, bg_y1), color, -1)
        cv2.putText(
            image, label,
            (x0 + 2, max(y0 - 3, th + 1)),
            font, font_scale, (255, 255, 255), 1, cv2.LINE_AA,
        )

    return image


def save_filtered_image(
    pdf_path: str,
    elements: list[dict],
    page_num: int,
    kind: str,
    scale: float = SCALE,
    out_folder: str = "filtered",
) -> None:
    
    import fitz  # PyMuPDF

    os.makedirs(out_folder, exist_ok=True)

    page_elems = [
        e for e in elements
        if e["page_num"] == page_num and e["kind"] == kind
    ]

    if not page_elems:
        print(f'[save_filtered_image] No "{kind}" elements on page {page_num}.')
        return

    # Render full page
    doc = fitz.open(pdf_path)
    pix = doc[page_num - 1].get_pixmap(
        matrix=fitz.Matrix(scale, scale), alpha=False
    )
    doc.close()

    real_page = np.frombuffer(pix.samples, np.uint8).reshape(
        pix.height, pix.width, pix.n
    )
    real_page = cv2.cvtColor(real_page, cv2.COLOR_RGB2BGR)

    # White canvas — same size
    canvas = np.ones((pix.height, pix.width, 3), dtype=np.uint8) * 255

    for e in page_elems:
        x0, y0, x1, y1 = e["bbox_img"]
        # Clamp to image bounds
        x0, x1 = max(x0, 0), min(x1, pix.width)
        y0, y1 = max(y0, 0), min(y1, pix.height)
        if x1 > x0 and y1 > y0:
            canvas[y0:y1, x0:x1] = real_page[y0:y1, x0:x1]

    out_path = os.path.join(out_folder, f"page_{page_num:03d}_{kind}.png")
    cv2.imwrite(out_path, canvas)
    print(f'Saved filtered image → {out_path}  ({len(page_elems)} "{kind}" elements)')


## PDFMiner

In [15]:
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox, LTTextLine, LTFigure, LTImage, LTLine


def _pdfminer_assign_reading_order(elements: list[dict]) -> list[dict]:
    from collections import defaultdict

    pages: dict = defaultdict(list)
    for e in elements:
        pages[e["page_num"]].append(e)

    counter = 1
    for page_num in sorted(pages):
        pg = pages[page_num]
        x_centers = [(e["bbox_pdf"][0] + e["bbox_pdf"][2]) / 2 for e in pg]
        page_mid  = (min(x_centers) + max(x_centers)) / 2
        for e in pg:
            xc = (e["bbox_pdf"][0] + e["bbox_pdf"][2]) / 2
            e["column"] = 0 if xc < page_mid else 1
        pg.sort(key=lambda e: (e["column"], -e["bbox_pdf"][3]))
        for e in pg:
            e["reading_order"] = counter
            counter += 1
    return elements


def _pdfminer_pdf_bbox_to_img(elements: list[dict], scale: float) -> list[dict]:
    for e in elements:
        x0, y0, x1, y1 = e["bbox_pdf"]
        ph = e["page_h"]
        e["bbox_img"] = (
            int(x0 * scale),
            int((ph - y1) * scale),
            int(x1 * scale),
            int((ph - y0) * scale),
        )
    return elements


def extract_elements_pdfminer(pdf_path: str, scale: float = SCALE) -> list[dict]:
    elements: list[dict] = []

    for page_num, page_layout in enumerate(extract_pages(pdf_path), start=1):
        page_h = page_layout.height

        for element in page_layout:
            bbox = element.bbox

            if isinstance(element, LTTextBox):
                for line in element:
                    if not isinstance(line, LTTextLine):
                        continue
                    text = line.get_text().strip()
                    if not text:
                        continue
                    elements.append({
                        "kind":      "text",
                        "page_num":  page_num,
                        "bbox_pdf":  line.bbox,
                        "page_h":    page_h,
                        "content":   text,
                    })

            # elif isinstance(element, LTImage):
            #     elements.append({
            #         "kind":      "image",
            #         "page_num":  page_num,
            #         "bbox_pdf":  bbox,
            #         "page_h":    page_h,
            #         "content":   "[IMAGE]",
            #     })

            elif isinstance(element, LTFigure):
                elements.append({
                    "kind":      "figure",
                    "page_num":  page_num,
                    "bbox_pdf":  bbox,
                    "page_h":    page_h,
                    "content":   "[FIGURE]",
                })

            # elif isinstance(element, LTLine):
            #     elements.append({
            #         "kind":      "line",
            #         "page_num":  page_num,
            #         "bbox_pdf":  bbox,
            #         "page_h":    page_h,
            #         "content":   "[LINE]",
            #     })

    _pdfminer_assign_reading_order(elements)
    _pdfminer_pdf_bbox_to_img(elements, scale)
    return elements


## Mistral OCR

In [ ]:
from mistralai import Mistral
from pydantic import BaseModel
from enum import Enum
from mistralai.extra import response_format_from_pydantic_model
from PIL import Image as PILImage, ImageDraw
import base64, fitz, os, io

client = Mistral(api_key=MISTRAL_API_KEY)

# ── Schema ─────────────────────────────────────────────
class ImageType(str, Enum):
    GRAPH = "graph"
    TEXT  = "text"
    TABLE = "table"
    IMAGE = "image"

class ImageAnnotation(BaseModel):
    image_type:  ImageType
    description: str

# ── Setup ──────────────────────────────────────────────
doc        = fitz.open(PDF_PATH)
out_folder = os.path.join(OUTPUT_ROOT, METHOD, "annotated_pages")
os.makedirs(out_folder, exist_ok=True)

# ── Process Pages ──────────────────────────────────────
for page_index in range(len(doc)):

    pix = doc[page_index].get_pixmap(matrix=fitz.Matrix(SCALE, SCALE), alpha=False)
    img_bytes = pix.tobytes("png")

    base64_image = base64.b64encode(img_bytes).decode("utf-8")

    response = client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "image_url",
            "image_url": f"data:image/png;base64,{base64_image}",
        },
        bbox_annotation_format=response_format_from_pydantic_model(ImageAnnotation),
        include_image_base64=False,
    )

    image = PILImage.open(io.BytesIO(img_bytes))
    draw = ImageDraw.Draw(image)

    w, h = image.size

    for page in response.pages:
        for img in page.images:

            def scale(val, max_val):
                return val * max_val if val <= 1 else val

            x1 = scale(img.top_left_x, w)
            y1 = scale(img.top_left_y, h)
            x2 = scale(img.bottom_right_x, w)
            y2 = scale(img.bottom_right_y, h)

            draw.rectangle([x1, y1, x2, y2], outline="red", width=3)

    out_path = os.path.join(out_folder, f"page_{page_index + 1:03d}_annotated.png")
    image.save(out_path)

    print(f"Saved → {out_path}")

doc.close()
print("Done.")

Saved → output\mistral\annotated_pages\page_001_annotated.png
Saved → output\mistral\annotated_pages\page_002_annotated.png
Saved → output\mistral\annotated_pages\page_003_annotated.png
Saved → output\mistral\annotated_pages\page_004_annotated.png
Saved → output\mistral\annotated_pages\page_005_annotated.png
Saved → output\mistral\annotated_pages\page_006_annotated.png
Saved → output\mistral\annotated_pages\page_007_annotated.png
Saved → output\mistral\annotated_pages\page_008_annotated.png
Saved → output\mistral\annotated_pages\page_009_annotated.png
Saved → output\mistral\annotated_pages\page_010_annotated.png
Saved → output\mistral\annotated_pages\page_011_annotated.png
Saved → output\mistral\annotated_pages\page_012_annotated.png
Done.


## Annotated Pages

In [16]:
import fitz
import csv


def render_annotated_pages(
    pdf_path: str,
    elements: list[dict],
    scale:    float = SCALE,
    out_folder: str = "annotated",
) -> None:
    
    os.makedirs(out_folder, exist_ok=True)
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        img = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        for e in [el for el in elements if el["page_num"] == page_num]:
            color = COLORS.get(e["kind"], (100, 100, 100))
            draw_bbox(img, e["bbox_img"], label=str(e["reading_order"]), color=color)

        out_path = os.path.join(out_folder, f"page_{page_num:03d}.png")
        cv2.imwrite(out_path, img)
        print(f"Saved → {out_path}")

    doc.close()
    print(f"Done. Annotated pages → {out_folder}")


def save_elements_csv(elements: list[dict], out_path: str = "elements.csv") -> None:
    
    fields = [
        "reading_order", "kind", "page_num",
        "pdf_x0", "pdf_y0", "pdf_x1", "pdf_y1",
        "img_x0", "img_y0", "img_x1", "img_y1",
        "content",
    ]
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for e in elements:
            x0, y0, x1, y1   = e["bbox_pdf"]
            ix0, iy0, ix1, iy1 = e["bbox_img"]
            writer.writerow({
                "reading_order": e["reading_order"],
                "kind":          e["kind"],
                "page_num":      e["page_num"],
                "pdf_x0":        round(float(x0), 2),
                "pdf_y0":        round(float(y0), 2),
                "pdf_x1":        round(float(x1), 2),
                "pdf_y1":        round(float(y1), 2),
                "img_x0":        ix0,
                "img_y0":        iy0,
                "img_x1":        ix1,
                "img_y1":        iy1,
                "content":       e.get("content", ""),
            })
    print(f"Saved CSV → {out_path}")


print("Render + CSV utilities loaded.")

Render + CSV utilities loaded.


## Execution

In [17]:
METHOD_OUT   = os.path.join(OUTPUT_ROOT, METHOD)
ANNOTATED_DIR = os.path.join(METHOD_OUT, "annotated_pages")
CSV_PATH      = os.path.join(METHOD_OUT, "elements.csv")

os.makedirs(METHOD_OUT, exist_ok=True)

if METHOD == "pdfminer":
    print("[pdfminer] Extracting elements …")
    elements = extract_elements_pdfminer(PDF_PATH, scale=SCALE)

elif METHOD == "mistral":
    print("[mistral] Calling OCR API …")
    elements = extract_elements_mistral(
        PDF_PATH,
        api_key=MISTRAL_API_KEY,
        pages=MISTRAL_PAGES,
        scale=SCALE,
    )

else:
    raise ValueError(f'Unknown METHOD "{METHOD}". Choose "pdfminer" or "mistral".')

print(f"Extracted {len(elements)} elements via [{METHOD}].")

render_annotated_pages(PDF_PATH, elements, scale=SCALE, out_folder=ANNOTATED_DIR)

save_elements_csv(elements, out_path=CSV_PATH)

[pdfminer] Extracting elements …
Extracted 488 elements via [pdfminer].
Saved → output\pdfminer\annotated_pages\page_001.png
Saved → output\pdfminer\annotated_pages\page_002.png
Saved → output\pdfminer\annotated_pages\page_003.png
Saved → output\pdfminer\annotated_pages\page_004.png
Saved → output\pdfminer\annotated_pages\page_005.png
Done. Annotated pages → output\pdfminer\annotated_pages
Saved CSV → output\pdfminer\elements.csv


## Preview

In [18]:
import pandas as pd

pd.set_option("display.max_colwidth", 60)
df = pd.read_csv(CSV_PATH)
print(f"Total elements: {len(df)}")
print("\nKind distribution:")
print(df["kind"].value_counts().to_string())
print()
display(df.head(30))

Total elements: 488

Kind distribution:
kind
text      481
figure      7



,reading_order,kind,page_num,pdf_x0,pdf_y0,pdf_x1,pdf_y1,img_x0,img_y0,img_x1,img_y1,content
0,83,text,1,99.19,749.14,496.09,763.49,206,163,1033,193,In Defense of RAG in the Era of Long-Context Language Mo...
1,1,text,1,129.92,714.81,166.59,726.77,270,239,347,264,Tan Yu
2,2,text,1,127.01,700.78,169.50,712.74,264,269,353,293,NVIDIA
3,3,text,1,93.30,686.83,203.21,698.79,194,298,423,323,"Santa Clara, California"
4,4,text,1,116.21,672.89,180.30,684.84,242,327,375,352,United States
5,5,text,1,103.42,659.44,193.08,671.40,215,355,402,380,tayu@nvidia.com
6,84,text,1,268.24,714.81,327.04,726.77,558,239,681,264,Anbang Xu
7,86,text,1,276.39,700.78,318.88,712.74,575,269,664,293,NVIDIA
8,88,text,1,242.69,686.83,352.59,698.79,505,298,734,323,"Santa Clara, California"
9,90,text,1,265.59,672.89,329.68,684.84,553,327,686,352,United States


## Filtered Image

In [ ]:
FILTER_PAGE = 1        # page number
FILTER_KIND = "figure"   # 'text' | 'image' | 'table' | 'graph' | 'figure' | 'line'

FILTERED_DIR = os.path.join(METHOD_OUT, "filtered_pages")

save_filtered_image(
    pdf_path   = PDF_PATH,
    elements   = elements,
    page_num   = FILTER_PAGE,
    kind       = FILTER_KIND,
    scale      = SCALE,
    out_folder = FILTERED_DIR,
)

Saved filtered image → output\pdfminer\filtered_pages\page_001_figure.png  (2 "figure" elements)


## save all kinds for all pages

In [20]:
all_kinds = df["kind"].unique().tolist()
all_pages = df["page_num"].unique().tolist()

print(f"Exporting kinds {all_kinds} across {len(all_pages)} page(s) …\n")

for page in sorted(all_pages):
    for kind in all_kinds:
        save_filtered_image(
            pdf_path   = PDF_PATH,
            elements   = elements,
            page_num   = page,
            kind       = kind,
            scale      = SCALE,
            out_folder = FILTERED_DIR,
        )

print("\nBatch export complete.")

Exporting kinds ['text', 'figure'] across 5 page(s) …

Saved filtered image → output\pdfminer\filtered_pages\page_001_text.png  (115 "text" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_001_figure.png  (2 "figure" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_002_text.png  (94 "text" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_002_figure.png  (1 "figure" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_003_text.png  (68 "text" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_003_figure.png  (2 "figure" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_004_text.png  (106 "text" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_004_figure.png  (2 "figure" elements)
Saved filtered image → output\pdfminer\filtered_pages\page_005_text.png  (98 "text" elements)
[save_filtered_image] No "figure" elements on page 5.

Batch export complete.
